# LFRic Iris data manipulation and visualisation practical

Let's apply what we've learned in Section 01 to Section 05 about data processing and visualisation of LFRic data in Iris and PyVista with the two exercises. The aim is to use the prompts to write the code yourself, but we have also provided a separate notebook containing the answers if you are stuck. All the information needed to write the code for this practical can be found in the notebooks in the first part of this practical. 

Note: This is delivered in JupyterLabs, but sometime the PyVista and GeoVista plotting is laggy in labs. If you prefer you can run in IPython.

## Step 0: Conda setup (required for regridding)

Run these commands in a **fresh terminal** before running notebook cells:

```bash
cd /Users/lb788/Documents/LFRic-Atm-Training/LFRic-Atmosphere-Training
conda create -n lfric-mesh python=3.12 -y
conda activate lfric-mesh
conda install -c conda-forge esmpy -y
# zsh-safe quoting for extras is important
python -m pip install -e '.[mesh_tutorials]'
python -m pip install jupyterlab
python -m pip show esmf-regrid
python -c "import esmf_regrid, ESMF; print('Imports OK')"
python -m jupyter lab --version
python -m ipykernel install --user --name lfric-mesh --display-name "Python (lfric-mesh)"
python -m jupyter lab
```

If `python -m pip show esmf-regrid` returns nothing, the package is not installed in that env.
If `import esmf_regrid` fails, rerun the quoted install command exactly as shown.

Then in Jupyter choose **Kernel -> Change Kernel -> Python (lfric-mesh)** and restart the kernel.

Reason: `esmf-regrid` needs the compiled ESMF backend (`esmpy`), which is installed from conda-forge.


In [1]:
# Notebook warning controls
import warnings

# Suppress repeated NumPy masked-array casting warnings from sample data loading.
warnings.filterwarnings(
    "ignore",
    message="invalid value encountered in cast",
    category=RuntimeWarning,
)
# Suppress Iris legacy date-precision future warning in tutorial output.
warnings.filterwarnings(
    "ignore",
    message="You are using legacy date precision for Iris units*",
    category=FutureWarning,
)

# Use microsecond date precision to align with upcoming Iris default behaviour.
try:
    import iris
    iris.FUTURE.date_microseconds = True
except Exception:
    pass

# Check that this kernel can run regridding
import importlib.util
import sys
print("Kernel Python:", sys.executable)
assert importlib.util.find_spec("ESMF") or importlib.util.find_spec("esmpy"), "ESMPy backend missing in this kernel"
assert importlib.util.find_spec("esmf_regrid"), "esmf_regrid missing in this kernel"
print("Backend check: OK")


Kernel Python: /Users/lb788/miniconda3/envs/lfric-mesh/bin/python
Backend check: OK


## Exercise 1 - Regrid LFRic to UM and plot data 

In this exercise you will take LFRic data, regrid it to UM data and then plot the differences

**Step 1** To begin, we need to import the neccesary packages that we will need for this exercise.

In [2]:
%matplotlib inline
import pyvista as pv
import geovista as gv
import geovista.theme
import iris.quickplot as qplt
import iris
from geovista import GeoPlotter
from esmf_regrid.experimental.unstructured_scheme import MeshToGridESMFRegridder, GridToMeshESMFRegridder
pv.set_jupyter_backend("static")
iris.FUTURE.datum_support = True  # avoids some warnings

The [pv_conversions](./pv_conversions.py) script contains two functions which convert LFRic cubes to pyvista objects. Load these two functions:

In [3]:
from support.pv_conversions import pv_from_lfric_cube

**Step 2** Use iris.load_cube to load in the LFRic data, and select the diagnostic 'surface_air_pressure'. Print the cube - is it a mesh or grid? \
    (hint: you will need to use PARSE_UGRID_ON_LOAD.context() )

In [4]:
# Define the location of the data and file names
data_path = '../example_data/'
lfric_path = data_path + 'u-ct674_20210324T0000Z_lf_ugrid.nc'
um_path = data_path + 'u-ct674_20210324T0000Z_um_latlon.nc'

# Continue here with importing the data... 


**Step 4** Select the first timestep from the data so we have a 2D cube

**Step 3** Transform the LFRic cube into a pyvista object using one of the functions from pv_conversions

**Step 5** Use GeoPlotter to plot the data using PyVista. You will need to use plotter.show() to display the plot.

You can now observe Surface Air Pressue on a 3D globe. 

**Step 6** Now, lets regrid some LFRic data onto a lat/lon grid

Use iris.load_cube to load the reference grid and print the cube. You can use the equivelent UM data loaded above for this. 

**Step 7** Create a regridder by using MeshToGridESMRegridder. Make sure the mesh and grid are the correct way round, consult the [regridding section](./04_Regridding.ipynb) of the notebooks for help.

**Step 8** Now, regrid the LFRic data using the regridder you created. Print the result - is your LFRic data regridded?

**Step 9** Using qplt.pcolormesh plot the regridded LFRic data

**Step 10** We can use the UM data loaded as reference for the regridding to compare to the reggrided LFRic data.\
Now select the first timestep of the UM data loaded in step 6

**Step 11** Now calulate the difference between the LFRic regridded data and native UM data

**Step 12** Now plot the original UM data and the regridded LFRic data and the difference side by side. \
(hint: use plt.subplot(1,3,1) and try adding a title and coastlines to the plot)